In [17]:
#to-do
#create an copy of original
#clean column names, string values in it
#drop duplicates
#check if all nA values need to be removed?
#create new cols that will help in future metrics

In [43]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/user_events_ecommerce_product_analytics.csv")
df_clean = df.copy()
df_clean.columns = df_clean.columns.str.strip().str.lower()
text_cols = ["event_type", "device", "traffic_source", "country", "city", "category"]
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()


#merge date and time into timestamp
df_clean["event_timestamp"] = pd.to_datetime(
    df_clean["event_date"].astype(str) + " " + df_clean["event_time"].astype(str),
    errors="coerce"
)


before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates()
after_dup = len(df_clean)
print("Duplicates removed:", before_dup - after_dup)

df_clean["price_present"] = df_clean["price"].notna().astype(int)

df_clean["invalid_price"] = df_clean["price"].notna() & (df_clean["price"] <= 0)

df_clean["bad_timestamp"] = df_clean["event_timestamp"].isna()

df_clean = df_clean.sort_values(["user_id", "session_id", "event_timestamp"])


df_clean["event_date_only"] = df_clean["event_timestamp"].dt.date
df_clean["event_hour"] = df_clean["event_timestamp"].dt.hour
df_clean["day_of_week"] = df_clean["event_timestamp"].dt.day_name()

# cleaned dataset saved to csv
df_clean.to_csv("../data/cleaned_user_events.csv", index=False)

print("Invalid prices found:", df_clean["invalid_price"].sum())
print("Bad timestamps found:", df_clean["bad_timestamp"].sum())
df_clean.head(5)

Duplicates removed: 0
Invalid prices found: 0
Bad timestamps found: 0


,user_id,session_id,event_date,event_time,event_type,product_id,category,price,device,traffic_source,country,city,event_timestamp,price_present,invalid_price,bad_timestamp,event_date_only,event_hour,day_of_week
220361,U100000,S1085471,2024-04-01,04:41:00,view,P4277,jeans,NaN,web,ads,germany,berlin,2024-04-01 04:41:00,0,False,False,2024-04-01,4,Monday
1890,U100000,S1303026,2024-01-26,02:09:00,visit,P6606,formal wear,NaN,web,organic,germany,berlin,2024-01-26 02:09:00,0,False,False,2024-01-26,2,Friday
227201,U100000,S1485879,2024-06-14,20:18:00,checkout,P9863,men pants,4791.33,web,ads,uk,london,2024-06-14 20:18:00,1,False,False,2024-06-14,20,Friday
138277,U100000,S2102407,2024-03-16,14:45:00,add_to_cart,P2593,women pants,3220.34,mobile,organic,india,mumbai,2024-03-16 14:45:00,1,False,False,2024-03-16,14,Saturday
170932,U100000,S2139541,2024-01-24,05:14:00,visit,P6605,jeans,NaN,mobile,ads,australia,melbourne,2024-01-24 05:14:00,0,False,False,2024-01-24,5,Wednesday


In [ ]:
#data cleaning steps performed:
#1. created a copy of original dataset, saved cleaned dataset to csv
#2. standardized column names and string values in text columns
#3. validated price data and confirmed that all price values are positive numbers, created new columns to identify missing and invalid prices
#4. checked for duplicate rows - none found
#5. preserved price value for event_type "view" and "visit" to maintain realistic event behavior.
#6. merged date and time into a single timestamp column, created new columns for date, hour, and day of week to facilitate future analysis


In [40]:
funnel_users = (
    df_clean.groupby("event_type")["user_id"]
    .nunique()
    .reset_index()
)

funnel_users.columns = ["event_type", "unique_users"]
funnel_users = funnel_users.sort_values(
    "unique_users", ascending=False
).reset_index(drop=True)

funnel_users["conversion_from_previous"] = (
    (funnel_users["unique_users"] / funnel_users["unique_users"].shift(1)
) * 100).round(2)

funnel_users

,event_type,unique_users,conversion_from_previous
0,view,11998,NaN
1,visit,11995,99.97
2,add_to_cart,11888,99.11
3,checkout,11533,97.01
4,purchase,10566,91.62


In [ ]:
#funnel insights:
#1. Doesn't give a clear picture as the drop off rates seem quite non-realisitc. Given this is synthetic data, it is possible. 
#2. The above funnel looks overly simplified because of the unique users count. The dataset is spread over 12 months, and users may have multiple sessions and events. So, the unique user count for each event type may not accurately reflect the true funnel behavior, as users can appear in multiple stages of the funnel across different sessions and time periods.
#2. Hence, I'll break down to granural funnel (per session)

In [ ]:
#session-funnel analysis
funnel_steps = [
    "visit",
    "view",
    "add_to_cart",
    "checkout",
    "purchase"
]

step_map = {step: i for i, step in enumerate(funnel_steps)}

df_funnel = df_clean[df_clean["event_type"].isin(funnel_steps)].copy()

df_funnel["step_num"] = df_funnel["event_type"].map(step_map)

# get max step reached per session
session_progress = (
    df_funnel.groupby("session_id")["step_num"]
    .max()
    .reset_index()
)

funnel_counts = []

total_sessions = len(session_progress)

for i, step in enumerate(funnel_steps):
    
    reached = (session_progress["step_num"] >= i).sum()
    
    funnel_counts.append({
        "event_type": step,
        "sessions_reached": reached
    })

funnel_df = pd.DataFrame(funnel_counts)

funnel_df["conversion_from_previous"] = (
    funnel_df["sessions_reached"]
    .pct_change()
    .add(1)
    .mul(100)
    .round(2)
)

funnel_df["dropoff_rate"] = (
    100 - funnel_df["conversion_from_previous"]
)

funnel_df

,event_type,sessions_reached,conversion_from_previous,dropoff_rate
0,visit,314284,NaN,NaN
1,view,221464,70.47,29.53
2,add_to_cart,121019,54.64,45.36
3,checkout,63765,52.69,47.31
4,purchase,25471,39.95,60.05


In [ ]:
#Granular funnel insights:
#1. Visit -> view: 70% is pretty decent, meaning product pages are engaging enough, Users are showing initial interest and traffic quality is good.
#2. View -> add_to_cart: drop off is 45%, meaning low product appeal or pricing issues. It could also be using treating the platform only to browse. (needs analysis later)
#3. Add_to_cart -> checkout: drop off is 47%, indicating any hidden costs, lengthy checkout process, or lack of payment options. (needs RCA)
#4. Checkout -> purchase: highest drop off - 60%, indicating checkout friction, payment issues, no offers, security concerns etc. (needs RCA)
#Mid-funnel degradation is a common issue in e-commerce, further analysis needs to be done to identify pain points and optimize the funnel.

In [50]:
#I'm gonna remove further unused columns to declutter the dataset i.e, "price_present", "invalid_price", "bad_timestamp" as they all came out to be 0 in the previous checks.
df_clean = df_clean.drop(columns=["price_present", "invalid_price", "bad_timestamp"])

In [ ]:
#mid-funnel RCA:
#Index([, 'category', 'price', 'device', 'traffic_source','country', 'city', 'event_timestamp', 'day_of_week'],
#the above can be considered potential factors impacting mid-funnel drop off.

#1. Traffic source analysis:
traffic_funnel = (
    df_funnel.groupby(["traffic_source", "event_type"])["session_id"]
    .nunique()
    .unstack()
)
traffic_funnel["visit_to_view_dropoff"] = (
    100 - (traffic_funnel["view"] / traffic_funnel["visit"] * 100).round(2)
)

traffic_funnel["view_to_cart_dropoff"] = (
    100 - (traffic_funnel["add_to_cart"] / traffic_funnel["view"] * 100).round(2)
)

traffic_funnel["cart_to_checkout_dropoff"] = (
    100 - (traffic_funnel["checkout"] / traffic_funnel["add_to_cart"] * 100).round(2)
)

traffic_funnel["checkout_to_purchase_dropoff"] = (
    100 - (traffic_funnel["purchase"] / traffic_funnel["checkout"] * 100).round(2)
)

traffic_funnel



NameError: name 'e' is not defined